# Multi-Channel Response Curves & Portfolio Budget Allocation: Practitioner's Template

**Author:** [Ryan Duecker](mailto:ryanduecker@google.com) | **Module:** `dxpoint`

This notebook provides a **production-ready, multi-channel boilerplate template** for growth marketers, media scientists, and econometrics practitioners.

While `practitioner_single_channel_template.ipynb` focuses on deep-dive diagnostics for an isolated tactic, this workbook is structured as an end-to-end operational pipeline for **multi-channel curve fitting and portfolio capital allocation**:
1. **Centralized Portfolio Configuration:** Configure your input data directory, CSV/TSV schema mappings, time aggregation rules, modeling defaults, and target portfolio budget.
2. **Directory-Based Multi-Dataset Ingestion:** Automatically scan a directory of channel files (e.g., `*.csv` or `*.tsv`), standardize dates, apply filters, and aggregate spend and return series to a unified daily or weekly cadence.
3. **Batch Multi-Channel Curve Fitting:** Fit `MarketingReturnCurve` models across all channels simultaneously with statistical uncertainty intervals and channel-specific adstock settings.
4. **Automated Multi-Channel Fit Diagnostics:** Evaluate goodness-of-fit ($R^2$, Adjusted $R^2$, MAPE, Residual Bias, and Extrapolation Risk) across every channel with automated health-check callouts.
5. **Parallel Incrementality Validation:** Optionally validate or calibrate multiple channel curves against causal lift studies (`PortfolioAllocator.validate_experiments`).
6. **Cross-Channel Tipping Points & Executive Curves:** Compare Peak Efficiency ($f''(x)=0$), Diminishing Returns ($f'(x)=\text{Target mROAS}$), and Optimal Scaling Zones across your entire media mix.
7. **Portfolio Budget Recommendation Working Area:** Input an overall budget number and optional channel guardrails to generate **executive recommendation tables (values)**, **what-if budget tier sensitivity matrices**, and **presentation-ready portfolio visualizations (plots)**.

## 1. Setup & Central Portfolio Configuration

Configure your directory path, file formatting, modeling hyperparameters, and overall portfolio budget target below. All downstream ingestion, curve fitting, diagnostics, and budget allocation cells inherit from this central block.

In [ ]:
import os
import glob
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from dxpoint import (
    MarketingReturnCurve,
    PortfolioAllocator,
    MultiChannelModel,
    CurveVisualizer,
)

# ==============================================================================
# 🛠️ PRACTITIONER CONFIGURATION: Directory, Fitting & Portfolio Budget Settings
# ==============================================================================

# 1. Directory & Dataset Schema Settings
DATA_DIR = "./channel_data"         # Directory containing per-channel CSV/TSV files
FILE_PATTERN = "*.csv"              # Glob pattern to match files (e.g., '*.csv' or '*.tsv')
FILE_SEPARATOR = ","                # Column delimiter (',' for CSV, '\t' for TSV)
DATE_COL = "date"                   # Date/timestamp column name
SPEND_COL = "spend"                 # Media spend column name
RETURN_COL = "return"               # Return / revenue / conversion value column name
CHANNEL_COL = None                  # Set column name if channel is inside the file; None infers from filename
FILTERS = []                        # Optional filters: [{'key': 'Market', 'value': 'US'}]
AGGREGATE_WEEKLY = False            # True: aggregate to weekly (W-MON); False: daily cadence

# 2. Global Curve Fitting Defaults
# FIT_METHOD options: 'auto', 'frequentist' (NLS with SEs & CIs), 'bayesian' (MCMC), 'gradient_descent'
FIT_METHOD = "frequentist"
DEFAULT_ADSTOCK_TYPE = "none"       # 'none', 'bounded', 'fixed', 'free'
DEFAULT_ADSTOCK_BOUNDS = (0.5, 5.0)
DEFAULT_ADSTOCK_FIXED_DAYS = 2.0
FIT_BASELINE = True                 # Estimate organic non-media baseline (beta_0) per channel
CONFIDENCE_LEVEL = 0.95             # Confidence / Credible interval width
TARGET_MROAS = 1.15                 # Hurdle rate for diminishing returns (or 1.0 / Target CPA)

# 3. Optional Per-Channel Overrides (Keyed by inferred channel name)
# Allows customizing adstock decay windows, hurdle rates, or current spend levels per tactic.
CHANNEL_OVERRIDES = {
    "Paid_Search": {
        "current_spend": 16_000.0,
    },
    "YouTube_Video": {
        "current_spend": 14_000.0,
    },
    "Paid_Social": {
        "current_spend": 12_000.0,
    },
    "Connected_TV": {
        "current_spend": 8_000.0,
    },
}

# 4. Portfolio Budget Optimization Target & Guardrails
# Overall budget number to allocate across all channels (in the same daily/weekly cadence as the data)
TOTAL_PORTFOLIO_BUDGET = 55_000.0

# Optional (min_spend, max_spend) business constraints per channel.
# Leave a channel out of CHANNEL_BOUNDS to allow unconstrained optimization [0, TOTAL_PORTFOLIO_BUDGET].
CHANNEL_BOUNDS = {
    "Paid_Search":   (4_000.0, 30_000.0),
    "YouTube_Video": (3_000.0, 30_000.0),
    "Paid_Social":   (2_000.0, 25_000.0),
    "Connected_TV":  (2_000.0, 25_000.0),
}

# 5. Optional Multi-Channel Incrementality Experiments (Set to None if no lift studies available)
EXPERIMENTS = {
    "Paid_Search": [
        {"name": "Search_Holdout_Q1", "spend": 12_000.0, "lift": 26_800.0, "se": 1_600.0},
    ],
    "YouTube_Video": [
        {"name": "YT_GeoLift_Q1", "spend": 10_000.0, "lift": 17_200.0, "se": 1_400.0},
        {"name": "YT_GeoLift_Q2", "spend": 18_000.0, "lift": 33_500.0, "se": 1_900.0},
    ],
}

## 2. Multi-CSV Directory Data Ingestion & Preparation

The ingestion pipeline scans `DATA_DIR` for all files matching `FILE_PATTERN` (e.g., one CSV per channel), applies optional filters, and aggregates timestamps to a consistent cadence.

> [!TIP]
> **Multi-Channel Data Hygiene:**
> - **Consistent Cadence:** Ensure all channel curves are fit on the **same time granularity** (e.g., all daily or all weekly) so `TOTAL_PORTFOLIO_BUDGET` represents a unified daily or weekly budget.
> - **Consistent Return Units:** Ensure the `return` column across all CSVs is measured in the same unit (e.g., gross revenue in USD, gross profit, or qualified conversions) so marginal returns ($f'(x)$) are directly comparable across channels.
> - **Out-of-the-Box Execution:** If `DATA_DIR` does not exist yet, the cell below automatically generates a realistic 4-channel suite of CSVs in a temporary directory and ingests them through the exact same file-reading pipeline.

In [ ]:
# ------------------------------------------------------------------------------
# 2. DIRECTORY DATA DISCOVERY & MULTI-CSV INGESTION
# ------------------------------------------------------------------------------

def _generate_sample_multichannel_csvs(target_dir: str, n_days: int = 120) -> str:
  """Generates 4 realistic channel CSV files in target_dir for out-of-the-box execution."""
  np.random.seed(42)
  dates = pd.date_range(start="2026-01-01", periods=n_days, freq="D")
  dow = dates.dayofweek.values

  # True underlying channel parameters for synthetic data generation
  channel_specs = {
      "Paid_Search": {
          "spend_range": (3000, 26000),
          "beta": 48000.0, "alpha": 0.95, "K": 10500.0, "baseline": 6000.0, "theta": 0.0, "noise": 950.0
      },
      "YouTube_Video": {
          "spend_range": (3000, 28000),
          "beta": 64000.0, "alpha": 1.65, "K": 15500.0, "baseline": 4500.0, "theta": 0.0, "noise": 1100.0
      },
      "Paid_Social": {
          "spend_range": (2000, 24000),
          "beta": 38000.0, "alpha": 1.35, "K": 11000.0, "baseline": 3200.0, "theta": 0.0, "noise": 850.0
      },
      "Connected_TV": {
          "spend_range": (2500, 25000),
          "beta": 52000.0, "alpha": 1.75, "K": 13500.0, "baseline": 2800.0, "theta": 0.0, "noise": 900.0
      },
  }

  for ch_name, spec in channel_specs.items():
    s_min, s_max = spec["spend_range"]
    raw_spend = np.random.uniform(s_min, s_max, size=n_days) * (1.0 + 0.10 * np.sin(dow / 7.0 * 2 * np.pi))
    
    s_adstock = np.zeros(n_days)
    for t in range(n_days):
      s_adstock[t] = raw_spend[t] + (spec["theta"] * s_adstock[t - 1] if t > 0 else 0.0)

    inc_ret = spec["beta"] * (s_adstock ** spec["alpha"]) / (spec["K"] ** spec["alpha"] + s_adstock ** spec["alpha"])
    obs_ret = np.maximum(0.0, spec["baseline"] + inc_ret + np.random.normal(0, spec["noise"], size=n_days))

    df_ch = pd.DataFrame({
        DATE_COL: dates.strftime("%Y-%m-%d"),
        "channel": ch_name,
        SPEND_COL: np.round(raw_spend, 2),
        RETURN_COL: np.round(obs_ret, 2),
    })
    df_ch.to_csv(os.path.join(target_dir, f"{ch_name}.csv"), index=False)
  return target_dir


# Check if user's DATA_DIR exists and has matching files; otherwise use generated sample directory
discovered_files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_PATTERN))) if os.path.isdir(DATA_DIR) else []
if len(discovered_files) == 0:
  _tmp_dir_obj = tempfile.TemporaryDirectory()
  active_data_dir = _generate_sample_multichannel_csvs(_tmp_dir_obj.name)
  discovered_files = sorted(glob.glob(os.path.join(active_data_dir, FILE_PATTERN)))
  print(f"ℹ️  No files found in '{DATA_DIR}'. Generated {len(discovered_files)} sample channel CSVs in temporary workspace.")
else:
  active_data_dir = DATA_DIR
  print(f"📂 Found {len(discovered_files)} channel dataset(s) in '{active_data_dir}'.")

# Read, filter, and aggregate each CSV file in the directory
channel_datasets = {}
ingestion_rows = []
cadence_label = "week" if AGGREGATE_WEEKLY else "day"

for filepath in discovered_files:
  df_raw = pd.read_csv(filepath, sep=FILE_SEPARATOR)
  df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])

  # Apply optional practitioner filters
  for f_rule in FILTERS:
    if f_rule["key"] in df_raw.columns:
      if f_rule["key"] != DATE_COL:
        df_raw = df_raw[df_raw[f_rule["key"]] == f_rule["value"]]
      else:
        df_raw = df_raw[df_raw[f_rule["key"]] <= f_rule["value"]]

  # Infer channel name from column or filename
  if CHANNEL_COL and CHANNEL_COL in df_raw.columns and len(df_raw[CHANNEL_COL].dropna()) > 0:
    ch_name = str(df_raw[CHANNEL_COL].dropna().iloc[0])
  else:
    ch_name = os.path.splitext(os.path.basename(filepath))[0]

  # Aggregate to weekly or daily cadence
  if AGGREGATE_WEEKLY:
    df_agg = (
        df_raw.groupby(pd.Grouper(key=DATE_COL, freq="W-MON", closed="left", label="left"))
        .agg({SPEND_COL: "sum", RETURN_COL: "sum"})
        .reset_index()
    )
  else:
    df_agg = (
        df_raw.groupby(DATE_COL)
        .agg({SPEND_COL: "sum", RETURN_COL: "sum"})
        .sort_values(DATE_COL)
        .reset_index()
    )

  spends_arr = df_agg[SPEND_COL].values.astype(float)
  returns_arr = df_agg[RETURN_COL].values.astype(float)

  # Determine current operational spend (from override or historical mean)
  ch_override = CHANNEL_OVERRIDES.get(ch_name, {})
  curr_spend = float(ch_override.get("current_spend", np.mean(spends_arr)))

  channel_datasets[ch_name] = {
      "df": df_agg,
      "spend": spends_arr,
      "return": returns_arr,
      "current_spend": curr_spend,
      "filepath": os.path.basename(filepath),
  }

  ingestion_rows.append({
      "Channel": ch_name,
      "Source File": os.path.basename(filepath),
      "Periods": len(df_agg),
      f"Mean Spend / {cadence_label}": f"${np.mean(spends_arr):,.0f}",
      f"Min–Max Spend / {cadence_label}": f"${np.min(spends_arr):,.0f} – ${np.max(spends_arr):,.0f}",
      f"Current Spend / {cadence_label}": f"${curr_spend:,.0f}",
      f"Mean Return / {cadence_label}": f"${np.mean(returns_arr):,.0f}",
  })

df_ingestion_summary = pd.DataFrame(ingestion_rows)
print("\n" + "=" * 95)
print(f"  INGESTED CHANNEL DATASETS SUMMARY ({len(channel_datasets)} Channels | Cadence: Per-{cadence_label.capitalize()})")
print("=" * 95)
df_ingestion_summary

## 3. Batch Curve Fitting Across Channels

We now loop through every ingested channel dataset and fit a non-linear Hill response curve with adstock carryover:

$$Return_{c,t} = \beta_{0,c} + \frac{\beta_c \cdot S_{c,t,\text{adstocked}}^{\alpha_c}}{K_c^{\alpha_c} + S_{c,t,\text{adstocked}}^{\alpha_c}}$$

*   **$\beta_c$ (Capacity Ceiling):** Maximum incremental return achievable on channel $c$.
*   **$\alpha_c$ (Curve Shape):** $\alpha_c > 1$ indicates an S-curve with a learning/warm-up threshold; $\alpha_c \le 1$ indicates a concave C-curve with immediate diminishing returns.
*   **$K_c$ (Half-Saturation):** Effective adstocked spend required to reach $50\%$ of $\beta_c$.
*   **$\theta_c$ (Adstock Retention):** Carryover decay rate from period to period.

In [ ]:
# ------------------------------------------------------------------------------
# 3. BATCH MULTI-CHANNEL CURVE FITTING
# ------------------------------------------------------------------------------

fitted_models = {}
model_list = []
param_rows = []

for ch_name, data in channel_datasets.items():
  ch_cfg = CHANNEL_OVERRIDES.get(ch_name, {})
  adstock_type = ch_cfg.get("adstock_type", DEFAULT_ADSTOCK_TYPE)
  adstock_bounds = ch_cfg.get("adstock_bounds", DEFAULT_ADSTOCK_BOUNDS)
  adstock_fixed = ch_cfg.get("adstock_fixed_days", DEFAULT_ADSTOCK_FIXED_DAYS)
  method = ch_cfg.get("fit_method", FIT_METHOD)

  model = MarketingReturnCurve.fit(
      spend_array=data["spend"],
      return_array=data["return"],
      channel_name=ch_name,
      method=method,
      adstock_type=adstock_type,
      adstock_bounds=adstock_bounds,
      adstock_fixed_days=adstock_fixed,
      fit_baseline=FIT_BASELINE,
      confidence_level=CONFIDENCE_LEVEL,
  )

  fitted_models[ch_name] = model
  model_list.append(model)

  s_info = model.summary()["parameters"]
  curve_type = "S-Curve (Warm-up)" if s_info["alpha"] > 1.0 else "C-Curve (Concave)"

  param_rows.append({
      "Channel": ch_name,
      "Curve Type": curve_type,
      "Capacity (β)": f"${s_info['beta']:,.0f}",
      "Shape (α)": f"{s_info['alpha']:.3f}",
      "Half-Saturation (K)": f"${s_info['K']:,.0f}",
      "Organic Baseline (β₀)": f"${s_info['baseline']:,.0f}",
      "Adstock (θ)": f"{s_info['theta']:.3f}",
      "Half-Life (Days)": f"{s_info['adstock_half_life_days']:.2f}",
  })

df_params = pd.DataFrame(param_rows)
print("\n" + "=" * 100)
print("  FITTED MULTI-CHANNEL RESPONSE CURVE PARAMETERS")
print("=" * 100)
df_params

## 4. Multi-Channel Fit Diagnostics & Automated Decision Callouts

In a multi-channel portfolio optimizer, a single poorly fitted or heavily extrapolated curve can distort budget allocations across the entire portfolio. Before running `PortfolioAllocator`, we screen every fitted curve against econometric quality gates:

*   **Explanatory Power ($R^2 \ge 0.60$):** Confirms media spend explains historical response variation.
*   **Commercial Precision ($\text{MAPE} \le 25\%$):** Confirms relative prediction errors are within acceptable bounds.
*   **Extrapolation Guardrail ($K \le 2 \times \max(\text{Spend})$):** Ensures the half-saturation point is anchored within observed historical spend rather than speculative extrapolation.

In [ ]:
# ------------------------------------------------------------------------------
# 4. MULTI-CHANNEL GOODNESS-OF-FIT TABLE & AUTOMATED DIAGNOSTIC CALLOUTS
# ------------------------------------------------------------------------------

fit_rows = []
portfolio_warnings = []

for ch_name, model in fitted_models.items():
  spends_arr = channel_datasets[ch_name]["spend"]
  m = model.evaluate_fit(verbose=False)

  r2 = m["r_squared"]
  adj_r2 = m["adj_r_squared"]
  mape = m["mape"]
  rmse = m["rmse"]
  bias_ratio = abs(m["residual_mean"]) / (m["residual_std"] + 1e-9)
  max_obs_spend = float(np.max(spends_arr))

  ch_flags = []
  if r2 < 0.60:
    ch_flags.append(f"Low R² ({r2:.2f} < 0.60)")
  if mape > 25.0:
    ch_flags.append(f"High MAPE ({mape:.1f}% > 25%)")
  if bias_ratio >= 0.10:
    ch_flags.append(f"Residual Bias ({m['residual_mean']:+.0f})")
  if model.K > 2.0 * max_obs_spend:
    ch_flags.append(f"K Extrapolation (K=${model.K:,.0f} > 2x Max Spend)")
  if model.alpha > 3.5:
    ch_flags.append(f"Steep Alpha ({model.alpha:.2f})")

  if len(ch_flags) == 0:
    health_badge = "✅ PASS"
    notes = "Strong fit & well-bracketed parameters"
  else:
    health_badge = "⚠️ REVIEW"
    notes = "; ".join(ch_flags)
    portfolio_warnings.append((ch_name, ch_flags))

  fit_rows.append({
      "Channel": ch_name,
      "Status": health_badge,
      "R²": f"{r2:.4f}",
      "Adj R²": f"{adj_r2:.4f}",
      "MAPE (%)": f"{mape:.2f}%",
      "RMSE": f"${rmse:,.0f}",
      "AIC": f"{m['aic']:,.1f}",
      "Diagnostic Callouts": notes,
  })

df_fit_diagnostics = pd.DataFrame(fit_rows)
print("\n" + "=" * 105)
print("  🩺 MULTI-CHANNEL GOODNESS-OF-FIT & DIAGNOSTIC HEALTH MATRIX")
print("=" * 105)
if len(portfolio_warnings) == 0:
  print("  ✅ ALL PORTFOLIO CHANNELS PASSED ECONOMETRIC DIAGNOSTIC GATES\n")
else:
  print(f"  ⚠️ ATTENTION: {len(portfolio_warnings)} CHANNEL(S) FLAGGED FOR PRACTITIONER REVIEW:")
  for flagged_ch, reasons in portfolio_warnings:
    print(f"     • [{flagged_ch}]: {', '.join(reasons)}")
    print("       ↳ Recommendation: Apply conservative upper spend bounds in CHANNEL_BOUNDS or calibrate with lift experiments.")
  print()

df_fit_diagnostics

## 5. Incrementality Experiment Validation Across Channels (Optional)

We instantiate our `PortfolioAllocator` with the fitted channel curves and attach any available causal lift studies (geo-holdouts, conversion lift tests) to verify that observational curves align with experimental truth prior to budget optimization.

In [ ]:
# ------------------------------------------------------------------------------
# 5. INITIALIZE PORTFOLIO ALLOCATOR & VALIDATE INCREMENTALITY EXPERIMENTS
# ------------------------------------------------------------------------------

allocator = PortfolioAllocator(model_list)

if EXPERIMENTS is not None and len(EXPERIMENTS) > 0:
  allocator.attach_experiments(EXPERIMENTS)
  val_results = allocator.validate_experiments(spend_is_raw=True, verbose=True)
  
  cal_summary = allocator.get_calibration_summary()
  cal_rows = [
      {
          "Channel": ch,
          "Experiments Attached": info["num_experiments"],
          "Alignment Verdict": info["verdict"],
          "Lift MAPE (%)": f"{info['mape']:.1f}%" if info["mape"] is not None else "N/A",
          "95% CI Coverage": f"{info['ci_coverage_pct']:.0f}%" if info["ci_coverage_pct"] is not None else "N/A",
      }
      for ch, info in cal_summary.items()
  ]
  df_cal_summary = pd.DataFrame(cal_rows)
else:
  print("No incrementality experiments configured. Proceeding with observational portfolio.")
  df_cal_summary = None

df_cal_summary

## 6. Cross-Channel Tipping Points & Individual Executive Curve Views

Before optimizing under a global budget constraint, we examine each channel's standalone calculus tipping points:
*   **Peak Efficiency ($x_{\text{inflection}}$ where $f''(x) = 0$):** Spend threshold required to exit the warm-up phase and reach maximum marginal ROAS.
*   **Diminishing Returns Cap ($x_{\text{diminishing}}$ where $f'(x) = \text{Target mROAS}$):** Maximum profitable spend before marginal returns fall below your hurdle rate.
*   **Standalone Headroom:** $\max(0, x_{\text{diminishing}} - x_{\text{current}})$.

In [ ]:
# ------------------------------------------------------------------------------
# 6.1: CROSS-CHANNEL TIPPING POINTS & HEADROOM SUMMARY TABLE
# ------------------------------------------------------------------------------

tipping_rows = []

for ch_name, model in fitted_models.items():
  curr_spend = channel_datasets[ch_name]["current_spend"]
  ch_target_mroas = CHANNEL_OVERRIDES.get(ch_name, {}).get("target_mroas", TARGET_MROAS)

  min_eff = model.get_minimal_marginal_cost_point() or 0.0
  max_prof = model.get_diminishing_returns_point(target_mroas=ch_target_mroas, warn_unreachable=False)
  curr_ret = float(model.predict_incremental_return(curr_spend))
  curr_mroas = float(model.predict_marginal_return(curr_spend))
  peak_mroas = float(model.predict_marginal_return(min_eff if min_eff > 0 else 1.0))

  if min_eff > 0 and curr_spend < min_eff:
    zone_status = "🟡 Warm-Up (Under-Scaled)"
  elif max_prof is not None and curr_spend <= max_prof:
    zone_status = "🟢 Optimal Scaling Zone"
  else:
    zone_status = "🔴 Over-Saturated (Sub-Hurdle)"

  headroom_val = (max_prof - curr_spend) if max_prof is not None else np.nan

  tipping_rows.append({
      "Channel": ch_name,
      "Current Spend": f"${curr_spend:,.0f}",
      "Peak Efficiency (f''=0)": f"${min_eff:,.0f}" if min_eff > 0 else "$0 (C-Curve)",
      "Peak mROAS": f"{peak_mroas:.2f}x",
      "Hurdle Cap (f'=Target)": f"${max_prof:,.0f}" if max_prof is not None else "Unreachable",
      "Standalone Headroom": f"{headroom_val:+,.0f}" if not np.isnan(headroom_val) else "N/A",
      "Current mROAS": f"{curr_mroas:.2f}x",
      "Current Status": zone_status,
  })

df_tipping_points = pd.DataFrame(tipping_rows)
print("\n" + "=" * 115)
print(f"  CROSS-CHANNEL TIPPING POINTS & STANDALONE HEADROOM MATRIX (Target Hurdle mROAS = {TARGET_MROAS})")
print("=" * 115)
df_tipping_points

In [ ]:
# ------------------------------------------------------------------------------
# 6.2: INDIVIDUAL CHANNEL 2-PANEL EXECUTIVE VIEWS
# ------------------------------------------------------------------------------
# Select a channel name (or loop over fitted_models) to inspect its 2-panel executive brief.
# Set RENDER_ALL_CHANNELS = True to display every channel's 2-panel plot sequentially.

RENDER_ALL_CHANNELS = False
SPOTLIGHT_CHANNEL = list(fitted_models.keys())[1] if len(fitted_models) > 1 else list(fitted_models.keys())[0]

channels_to_plot = list(fitted_models.keys()) if RENDER_ALL_CHANNELS else [SPOTLIGHT_CHANNEL]

for ch_name in channels_to_plot:
  ch_model = fitted_models[ch_name]
  ch_data = channel_datasets[ch_name]
  ch_inc_returns = ch_data["return"] - ch_model.baseline if ch_model.baseline > 0 else ch_data["return"]

  fig_exec = ch_model.plot_executive_view(
      target_mroas=CHANNEL_OVERRIDES.get(ch_name, {}).get("target_mroas", TARGET_MROAS),
      current_spend=ch_data["current_spend"],
      show_intervals=True,
      scatter=(ch_data["spend"], ch_inc_returns),
      figsize=(16, 6.5),
      show=True,
  )

## 7. Portfolio Budget Optimization & Recommendations Working Area

This section serves as your interactive **Practitioner Working Area** for generating media budget recommendations based on an overall portfolio budget number.

### How `PortfolioAllocator` Optimizes the Mix:
Given a total budget $B_{\text{total}}$ and per-channel min/max guardrails $[L_c, U_c]$, `PortfolioAllocator.allocate_budget()` uses constrained non-linear optimization (Sequential Least Squares Programming with multi-start inflection & Dirichlet initialization) to solve:

$$\max_{S_1, \dots, S_C} \sum_{c=1}^{C} f_c(S_c) \quad \text{subject to} \quad \sum_{c=1}^{C} S_c = B_{\text{total}}, \quad L_c \le S_c \le U_c$$

At the optimal allocation, **marginal ROAS is equalized** across all unbounded channels ($\frac{\partial f_1}{\partial S_1} = \frac{\partial f_2}{\partial S_2} = \dots = \lambda^*$), ensuring every marketing dollar is deployed where it yields the highest incremental return.

In [ ]:
# ==============================================================================
# 🎯 7.1: PRACTITIONER WORKING AREA — SET OVERALL BUDGET & RUN OPTIMIZATION
# ==============================================================================

# Modify WORKING_TOTAL_BUDGET or WORKING_CHANNEL_BOUNDS here to test new scenarios:
WORKING_TOTAL_BUDGET = TOTAL_PORTFOLIO_BUDGET
WORKING_CHANNEL_BOUNDS = CHANNEL_BOUNDS

# Run portfolio optimization
opt_result = allocator.allocate_budget(
    total_budget=WORKING_TOTAL_BUDGET,
    channel_bounds=WORKING_CHANNEL_BOUNDS,
)

# Compute baseline metrics at current spend vs. proportional scaling to WORKING_TOTAL_BUDGET
current_spends = {ch: channel_datasets[ch]["current_spend"] for ch in fitted_models}
total_current_spend = sum(current_spends.values())
current_returns = {ch: float(fitted_models[ch].predict_incremental_return(current_spends[ch])) for ch in fitted_models}
total_current_return = sum(current_returns.values())

# Proportional scaling benchmark (if current mix were scaled linearly to WORKING_TOTAL_BUDGET)
scale_factor = WORKING_TOTAL_BUDGET / total_current_spend if total_current_spend > 0 else 1.0
prop_spends = {ch: current_spends[ch] * scale_factor for ch in fitted_models}
prop_returns = {ch: float(fitted_models[ch].predict_incremental_return(prop_spends[ch])) for ch in fitted_models}
total_prop_return = sum(prop_returns.values())

opt_spends = opt_result["allocation"]
opt_returns = opt_result["channel_returns"]
opt_mroas = opt_result["marginal_roas"]
total_opt_return = opt_result["expected_total_return"]

# Build Actionable Channel Recommendation Table
rec_rows = []
for ch_name, model in fitted_models.items():
  c_spend = current_spends[ch_name]
  o_spend = opt_spends[ch_name]
  s_delta = o_spend - c_spend
  s_pct = (s_delta / c_spend * 100.0) if c_spend > 0 else 0.0
  share_pct = (o_spend / WORKING_TOTAL_BUDGET * 100.0) if WORKING_TOTAL_BUDGET > 0 else 0.0

  c_ret = current_returns[ch_name]
  o_ret = opt_returns[ch_name]
  r_delta = o_ret - c_ret

  avg_roas = o_ret / o_spend if o_spend > 0 else 0.0
  c_mr = float(model.predict_marginal_return(c_spend))
  o_mr = opt_mroas[ch_name]

  if s_pct > 5.0:
    action = f"🟢 SCALE UP ({s_pct:+.1f}%)"
  elif s_pct < -5.0:
    action = f"🟠 SCALE BACK ({s_pct:+.1f}%)"
  else:
    action = f"🔵 MAINTAIN ({s_pct:+.1f}%)"

  rec_rows.append({
      "Channel": ch_name,
      "Current Spend": f"${c_spend:,.0f}",
      "Recommended Spend": f"${o_spend:,.0f}",
      "Spend Delta ($)": f"{s_delta:+,.0f}",
      "Spend Shift (%)": f"{s_pct:+.1f}%",
      "Portfolio Share": f"{share_pct:.1f}%",
      "Current Return": f"${c_ret:,.0f}",
      "Projected Return": f"${o_ret:,.0f}",
      "Return Lift ($)": f"{r_delta:+,.0f}",
      "Avg ROAS": f"{avg_roas:.2f}x",
      "Current mROAS": f"{c_mr:.2f}x",
      "Optimal mROAS": f"{o_mr:.2f}x",
      "Recommendation": action,
  })

df_recommendations = pd.DataFrame(rec_rows)

# Print Executive Portfolio Summary Callout
budget_delta = WORKING_TOTAL_BUDGET - total_current_spend
return_lift_vs_curr = total_opt_return - total_current_return
return_lift_vs_prop = total_opt_return - total_prop_return
eq_mroas = float(np.median(list(opt_mroas.values())))

print("\n" + "=" * 115)
print(f"  📊 EXECUTIVE PORTFOLIO RECOMMENDATION SUMMARY (Overall Budget: ${WORKING_TOTAL_BUDGET:,.0f} / {cadence_label})")
print("=" * 115)
print(f"  • Optimization Convergence : {'✅ SUCCESS' if opt_result['success'] else '⚠️ ' + opt_result['message']}")
print(f"  • Total Portfolio Budget   : ${WORKING_TOTAL_BUDGET:>12,.0f}  (vs. Current Spend: ${total_current_spend:,.0f} | Delta: {budget_delta:+,.0f})")
print(f"  • Expected Incremental Ret : ${total_opt_return:>12,.0f}  (vs. Current Return: ${total_current_return:,.0f} | Lift:  {return_lift_vs_curr:+,.0f})")
print(f"  • Pure Reallocation Alpha  : ${return_lift_vs_prop:>+12,.0f}  (Extra return gained over naive proportional scaling at ${WORKING_TOTAL_BUDGET:,.0f})")
print(f"  • Portfolio Blended ROAS   : {opt_result['overall_roas']:>12.2f}x (vs. Current Blended ROAS: {total_current_return / total_current_spend:.2f}x)")
print(f"  • Equilibrium Marginal ROAS: {eq_mroas:>12.2f}x (Target Hurdle Rate: {TARGET_MROAS:.2f}x)")
print("=" * 115)

df_recommendations

In [ ]:
# ------------------------------------------------------------------------------
# 7.2: MULTI-SCENARIO BUDGET SENSITIVITY MATRIX (WHAT-IF BUDGET TIERS)
# ------------------------------------------------------------------------------
# Evaluates optimal channel allocations across multiple total budget scenarios
# so leadership can see how incremental funding is deployed across channels.

budget_multipliers = [0.70, 0.85, 1.00, 1.15, 1.30, 1.50]
tier_budgets = sorted(list(set([round(total_current_spend, -2)] + [round(WORKING_TOTAL_BUDGET * m, -2) for m in budget_multipliers])))

tier_rows = []
for b_tier in tier_budgets:
  res_tier = allocator.allocate_budget(total_budget=b_tier, channel_bounds=WORKING_CHANNEL_BOUNDS)
  eq_mr = float(np.median(list(res_tier["marginal_roas"].values())))

  if abs(b_tier - WORKING_TOTAL_BUDGET) < 1.0:
    tier_label = f"${b_tier:,.0f} (Target Budget)"
  elif abs(b_tier - total_current_spend) < 1.0:
    tier_label = f"${b_tier:,.0f} (Current Total)"
  else:
    pct_vs_target = (b_tier / WORKING_TOTAL_BUDGET - 1.0) * 100.0
    tier_label = f"${b_tier:,.0f} ({pct_vs_target:+.0f}% vs Target)"

  row_dict = {
      "Budget Scenario": tier_label,
      "Expected Return": f"${res_tier['expected_total_return']:,.0f}",
      "Blended ROAS": f"{res_tier['overall_roas']:.2f}x",
      "Equilibrium mROAS": f"{eq_mr:.2f}x",
  }
  for ch_name in fitted_models:
    ch_alloc = res_tier["allocation"][ch_name]
    row_dict[f"{ch_name} ($)"] = f"${ch_alloc:,.0f} ({ch_alloc / b_tier * 100:.0f}%)"
  tier_rows.append(row_dict)

df_budget_tiers = pd.DataFrame(tier_rows)
print("\n" + "=" * 115)
print("  WHAT-IF PORTFOLIO BUDGET SENSITIVITY MATRIX")
print("=" * 115)
df_budget_tiers

### 7.3 Executive Portfolio Visualizations (Plots)

Below we generate three presentation-ready matplotlib figures for executive budget reviews:
1. **Figure 1 (`fig_overlay`): 2-Panel Cross-Channel Response & Marginal Efficiency Overlay** — Shows all channel saturation curves on the left and marginal return curves on the right, highlighting the shift from Current Spend ($\circ$) to Recommended Spend ($\bullet$) and the equalized portfolio marginal ROAS line.
2. **Figure 2 (`fig_reallocation`): 3-Panel Executive Budget Reallocation Dashboard** — Compares Current vs. Recommended Spend by channel, the dollar reallocation waterfall ($\Delta \$$), and Expected Return by channel.
3. **Figure 3 (`fig_trajectory`): 2-Panel Optimal Scale Mix & Portfolio Frontier** — Displays how optimal channel budget shares evolve across total portfolio budget levels and where the overall portfolio hits the target hurdle rate.

In [ ]:
# ------------------------------------------------------------------------------
# 7.3A: FIGURE 1 — 2-PANEL CROSS-CHANNEL SATURATION & MARGINAL ROAS OVERLAY
# ------------------------------------------------------------------------------

PALETTE = ["#4285F4", "#7C3AED", "#0F9D58", "#F4B400", "#EA4335", "#00ACC1"]
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Roboto", "Google Sans", "Open Sans", "Arial", "DejaVu Sans"]

def _fmt_curr(x, pos=None):
  val = float(x)
  if abs(val) >= 1e6:
    return f"${val*1e-6:.1f}M"
  elif abs(val) >= 1e3:
    return f"${val*1e-3:.0f}k"
  return f"${val:,.0f}"

max_plot_spend = max(max(opt_spends.values()), max(current_spends.values())) * 1.45
x_grid = np.linspace(100.0, max_plot_spend, 400)

fig_overlay, (ax_sat, ax_mr) = plt.subplots(1, 2, figsize=(16, 6.8), facecolor="white")
ax_sat.set_facecolor("white")
ax_mr.set_facecolor("white")

for idx, (ch_name, model) in enumerate(fitted_models.items()):
  color = PALETTE[idx % len(PALETTE)]
  y_ret = model.predict_incremental_return(x_grid)
  y_mr = model.predict_marginal_return(x_grid)

  c_s = current_spends[ch_name]
  o_s = opt_spends[ch_name]
  c_r = current_returns[ch_name]
  o_r = opt_returns[ch_name]
  c_m = float(model.predict_marginal_return(c_s))
  o_m = opt_mroas[ch_name]

  # Panel 1: Saturation Curve + Current (open circle) + Optimal (filled circle)
  ax_sat.plot(x_grid, y_ret, color=color, lw=2.8, label=f"{ch_name} (Optimal: ${o_s:,.0f})", zorder=3)
  ax_sat.scatter([c_s], [c_r], color="white", edgecolors=color, s=75, lw=2.0, zorder=5)
  ax_sat.scatter([o_s], [o_r], color=color, edgecolors="#202124", s=110, lw=1.2, zorder=6)

  # Panel 2: Marginal Return Curve + Current (open circle) + Optimal (filled circle)
  ax_mr.plot(x_grid, y_mr, color=color, lw=2.5, label=f"{ch_name} (mROAS: {o_m:.2f}x)", zorder=3)
  ax_mr.scatter([c_s], [c_m], color="white", edgecolors=color, s=75, lw=2.0, zorder=5)
  ax_mr.scatter([o_s], [o_m], color=color, edgecolors="#202124", s=110, lw=1.2, zorder=6)

# Dummy legend handles for marker key
ax_sat.scatter([], [], color="white", edgecolors="#5F6368", s=70, lw=2.0, label="Current Spend")
ax_sat.scatter([], [], color="#5F6368", edgecolors="#202124", s=95, lw=1.2, label="Recommended Spend")

# Equilibrium mROAS & Target Hurdle lines on Panel 2
ax_mr.axhline(eq_mroas, color="#7C3AED", ls="--", lw=1.8, alpha=0.85, label="Equalized Portfolio mROAS", zorder=2)
ax_mr.axhline(TARGET_MROAS, color="#EA4335", ls=":", lw=1.6, alpha=0.80, label="Target Hurdle Rate", zorder=2)

# Styling Panel 1
ax_sat.set_title("1. Multi-Channel Saturation Curves (Current vs. Optimal)", fontsize=13, fontweight="bold", loc="left", pad=12, color="#202124")
ax_sat.set_xlabel(f"Channel Spend / {cadence_label.capitalize()}", fontsize=11, color="#5F6368")
ax_sat.set_ylabel("Incremental Return ($)", fontsize=11, color="#202124")
ax_sat.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_sat.yaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_sat.set_xlim(0, max_plot_spend)
ax_sat.set_ylim(bottom=0)
ax_sat.grid(True, alpha=0.15, color="#5F6368")
ax_sat.spines["top"].set_visible(False)
ax_sat.spines["right"].set_visible(False)
ax_sat.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.95, fontsize=9)

# Styling Panel 2
ax_mr.set_title("2. Marginal Return Equalization Across Channels", fontsize=13, fontweight="bold", loc="left", pad=12, color="#202124")
ax_mr.set_xlabel(f"Channel Spend / {cadence_label.capitalize()}", fontsize=11, color="#5F6368")
ax_mr.set_ylabel("Marginal ROAS (mROAS)", fontsize=11, color="#202124")
ax_mr.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_mr.set_xlim(0, max_plot_spend)
ax_mr.set_ylim(0, max(max(opt_mroas.values()) * 1.8, TARGET_MROAS * 2.0, 3.5))
ax_mr.grid(True, alpha=0.15, color="#5F6368")
ax_mr.spines["top"].set_visible(False)
ax_mr.spines["right"].set_visible(False)
ax_mr.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.95, fontsize=9)

fig_overlay.suptitle(
    f"Cross-Channel Response & Marginal Equilibrium (Total Budget: ${WORKING_TOTAL_BUDGET:,.0f})",
    fontsize=15, fontweight="bold", color="#202124", y=0.98
)
fig_overlay.text(
    0.5, 0.93,
    f"Expected Incremental Return: ${total_opt_return:,.0f} (Blended ROAS: {opt_result['overall_roas']:.2f}x) • Equilibrium mROAS: {eq_mroas:.2f}x",
    ha="center", fontsize=10.5, color="#5F6368", style="italic", parse_math=False
)
plt.tight_layout(rect=[0, 0, 1, 0.91])
plt.show()

In [ ]:
# ------------------------------------------------------------------------------
# 7.3B: FIGURE 2 — 3-PANEL EXECUTIVE BUDGET REALLOCATION & RETURN LIFT DASHBOARD
# ------------------------------------------------------------------------------

ch_labels = list(fitted_models.keys())
y_pos = np.arange(len(ch_labels))
curr_vals = np.array([current_spends[ch] for ch in ch_labels])
opt_vals = np.array([opt_spends[ch] for ch in ch_labels])
delta_vals = opt_vals - curr_vals
curr_ret_vals = np.array([current_returns[ch] for ch in ch_labels])
opt_ret_vals = np.array([opt_returns[ch] for ch in ch_labels])

fig_realloc, (ax_b1, ax_b2, ax_b3) = plt.subplots(1, 3, figsize=(18, 5.8), facecolor="white")
bar_h = 0.36

# Panel 1: Current vs Recommended Spend
ax_b1.set_facecolor("white")
ax_b1.barh(y_pos + bar_h / 2, curr_vals, height=bar_h, color="#9AA0A6", alpha=0.75, label="Current Spend")
ax_b1.barh(y_pos - bar_h / 2, opt_vals, height=bar_h, color="#4285F4", label="Recommended Spend")
for i, (cv, ov) in enumerate(zip(curr_vals, opt_vals)):
  ax_b1.text(ov + max(opt_vals) * 0.02, i - bar_h / 2, f"${ov:,.0f}", va="center", fontsize=9, fontweight="bold", color="#1A73E8")
ax_b1.set_yticks(y_pos)
ax_b1.set_yticklabels(ch_labels, fontsize=10, fontweight="500")
ax_b1.invert_yaxis()
ax_b1.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_b1.set_xlim(0, max(max(curr_vals), max(opt_vals)) * 1.25)
ax_b1.set_title("A. Current vs. Recommended Spend", fontsize=12.5, fontweight="bold", loc="left", pad=10)
ax_b1.set_xlabel("Spend ($)", fontsize=10, color="#5F6368")
ax_b1.legend(loc="lower right", frameon=True, facecolor="white", fontsize=8.5)
ax_b1.grid(True, axis="x", alpha=0.15)
ax_b1.spines["top"].set_visible(False)
ax_b1.spines["right"].set_visible(False)

# Panel 2: Spend Shift Delta ($)
ax_b2.set_facecolor("white")
delta_colors = ["#34A853" if d >= 0 else "#EA4335" for d in delta_vals]
ax_b2.barh(y_pos, delta_vals, height=0.5, color=delta_colors, alpha=0.85)
ax_b2.axvline(0, color="#202124", lw=1.2)
max_abs_delta = max(np.max(np.abs(delta_vals)), 1000.0)
for i, d in enumerate(delta_vals):
  offset = max_abs_delta * 0.04 if d >= 0 else -max_abs_delta * 0.04
  ha_align = "left" if d >= 0 else "right"
  ax_b2.text(d + offset, i, f"{d:+,.0f}", va="center", ha=ha_align, fontsize=9, fontweight="bold", color="#202124")
ax_b2.set_yticks(y_pos)
ax_b2.set_yticklabels(ch_labels, fontsize=10, fontweight="500")
ax_b2.invert_yaxis()
ax_b2.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_b2.set_xlim(-max_abs_delta * 1.35, max_abs_delta * 1.35)
ax_b2.set_title("B. Recommended Budget Shift (Δ$)", fontsize=12.5, fontweight="bold", loc="left", pad=10)
ax_b2.set_xlabel("Spend Change ($)", fontsize=10, color="#5F6368")
ax_b2.grid(True, axis="x", alpha=0.15)
ax_b2.spines["top"].set_visible(False)
ax_b2.spines["right"].set_visible(False)

# Panel 3: Expected Incremental Return by Channel
ax_b3.set_facecolor("white")
ax_b3.barh(y_pos + bar_h / 2, curr_ret_vals, height=bar_h, color="#9AA0A6", alpha=0.75, label="Current Return")
ax_b3.barh(y_pos - bar_h / 2, opt_ret_vals, height=bar_h, color="#7C3AED", label="Projected Return")
for i, (cr, or_val) in enumerate(zip(curr_ret_vals, opt_ret_vals)):
  ax_b3.text(or_val + max(opt_ret_vals) * 0.02, i - bar_h / 2, f"${or_val:,.0f}", va="center", fontsize=9, fontweight="bold", color="#7C3AED")
ax_b3.set_yticks(y_pos)
ax_b3.set_yticklabels(ch_labels, fontsize=10, fontweight="500")
ax_b3.invert_yaxis()
ax_b3.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_b3.set_xlim(0, max(max(curr_ret_vals), max(opt_ret_vals)) * 1.25)
ax_b3.set_title("C. Expected Incremental Return", fontsize=12.5, fontweight="bold", loc="left", pad=10)
ax_b3.set_xlabel("Incremental Return ($)", fontsize=10, color="#5F6368")
ax_b3.legend(loc="lower right", frameon=True, facecolor="white", fontsize=8.5)
ax_b3.grid(True, axis="x", alpha=0.15)
ax_b3.spines["top"].set_visible(False)
ax_b3.spines["right"].set_visible(False)

fig_realloc.suptitle(
    f"Executive Budget Reallocation Summary (Target Portfolio Budget: ${WORKING_TOTAL_BUDGET:,.0f})",
    fontsize=14.5, fontweight="bold", color="#202124", y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------------------
# 7.3C: FIGURE 3 — OPTIMAL SCALE MIX TRAJECTORY & PORTFOLIO FRONTIER
# ------------------------------------------------------------------------------
# Sweeps total portfolio budget from 30% to 170% of WORKING_TOTAL_BUDGET to show
# how optimal channel weights evolve and where the portfolio hits target mROAS.

min_possible_budget = sum(b[0] for b in WORKING_CHANNEL_BOUNDS.values()) if WORKING_CHANNEL_BOUNDS else 0.0
max_possible_budget = sum(b[1] for b in WORKING_CHANNEL_BOUNDS.values()) if WORKING_CHANNEL_BOUNDS else WORKING_TOTAL_BUDGET * 2.0
sweep_min = max(min_possible_budget * 1.05, WORKING_TOTAL_BUDGET * 0.35)
sweep_max = min(max_possible_budget * 0.95, WORKING_TOTAL_BUDGET * 1.70)

budget_grid = np.linspace(sweep_min, sweep_max, 25)
alloc_history = {ch: [] for ch in ch_labels}
portfolio_returns = []
portfolio_eq_mroas = []

for b_val in budget_grid:
  r_sweep = allocator.allocate_budget(total_budget=float(b_val), channel_bounds=WORKING_CHANNEL_BOUNDS)
  for ch in ch_labels:
    alloc_history[ch].append(r_sweep["allocation"][ch])
  portfolio_returns.append(r_sweep["expected_total_return"])
  portfolio_eq_mroas.append(float(np.median(list(r_sweep["marginal_roas"].values()))))

fig_sweep, (ax_s1, ax_s2) = plt.subplots(1, 2, figsize=(16, 6.2), facecolor="white")

# Panel 1: Stacked Area Optimal Mix
ax_s1.set_facecolor("white")
ax_s1.stackplot(
    budget_grid,
    [alloc_history[ch] for ch in ch_labels],
    labels=ch_labels,
    colors=PALETTE[:len(ch_labels)],
    alpha=0.82,
)
ax_s1.axvline(WORKING_TOTAL_BUDGET, color="#202124", ls="--", lw=2.0, label=f"Target Budget (${WORKING_TOTAL_BUDGET:,.0f})")
ax_s1.set_title("1. Optimal Channel Scale Mix Across Total Budget Levels", fontsize=13, fontweight="bold", loc="left", pad=10)
ax_s1.set_xlabel("Total Portfolio Budget ($)", fontsize=11, color="#5F6368")
ax_s1.set_ylabel("Optimal Channel Allocation ($)", fontsize=11, color="#202124")
ax_s1.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_s1.yaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_s1.set_xlim(sweep_min, sweep_max)
ax_s1.set_ylim(0, sweep_max)
ax_s1.legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.95, fontsize=9)
ax_s1.grid(True, alpha=0.15)
ax_s1.spines["top"].set_visible(False)
ax_s1.spines["right"].set_visible(False)

# Panel 2: Portfolio Equilibrium Marginal ROAS vs Total Budget
ax_s2.set_facecolor("white")
ax_s2.plot(budget_grid, portfolio_eq_mroas, color="#7C3AED", lw=3.0, label="Portfolio Equilibrium mROAS")
ax_s2.axhline(TARGET_MROAS, color="#EA4335", ls="--", lw=1.6, label="Target Hurdle Rate")
ax_s2.axvline(WORKING_TOTAL_BUDGET, color="#202124", ls="--", lw=2.0)
ax_s2.scatter([WORKING_TOTAL_BUDGET], [eq_mroas], color="#7C3AED", edgecolors="#202124", s=120, lw=1.5, zorder=5)
ax_s2.annotate(
    f"Target Budget: ${WORKING_TOTAL_BUDGET:,.0f}\nEquilibrium mROAS: {eq_mroas:.2f}x",
    xy=(WORKING_TOTAL_BUDGET, eq_mroas),
    xytext=(15, 20), textcoords="offset points",
    fontsize=9, fontweight="bold", color="#202124",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#F3E8FF", edgecolor="#7C3AED", alpha=1.0, zorder=10),
    arrowprops=dict(arrowstyle="->", color="#202124", lw=1.2, zorder=10),
    zorder=10,
)
ax_s2.set_title("2. Portfolio Marginal Efficiency Frontier", fontsize=13, fontweight="bold", loc="left", pad=10)
ax_s2.set_xlabel("Total Portfolio Budget ($)", fontsize=11, color="#5F6368")
ax_s2.set_ylabel("Equilibrium Marginal ROAS", fontsize=11, color="#202124")
ax_s2.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt_curr))
ax_s2.set_xlim(sweep_min, sweep_max)
ax_s2.set_ylim(0, max(max(portfolio_eq_mroas) * 1.25, TARGET_MROAS * 1.5))
ax_s2.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.95, fontsize=9)
ax_s2.grid(True, alpha=0.15)
ax_s2.spines["top"].set_visible(False)
ax_s2.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## 8. Practitioner Governance Checklist & Next Steps

Before presenting multi-channel budget recommendations or executing cross-channel reallocations, verify the following governance gates:

- [ ] **1. Time-Unit & Currency Consistency:** Confirmed that all ingested CSV files share the exact same time cadence (all daily or all weekly) and that `return` is measured in identical financial units across channels.
- [ ] **2. Individual Curve Fit Quality:** Verified in Section 4 that every channel passes $R^2 \ge 0.60$, $\text{MAPE} \le 25\%$, and unbiased residual checks.
- [ ] **3. Extrapolation Guardrails (`CHANNEL_BOUNDS`):** Any channel whose half-saturation $K$ exceeds $2\times$ historical max spend is constrained in `CHANNEL_BOUNDS` so the optimizer does not over-allocate into ungrounded territory.
- [ ] **4. Causal Lift Alignment:** Where holdout or geo-lift tests exist, verified alignment in Section 5 (`allocator.validate_experiments()`) or calibrated the channel using `MarketingReturnCurve.fit_bayesian(..., calibration_experiments=...)`.
- [ ] **5. Joint Multi-Channel Collinearity Check:** If channels are highly correlated in spend timing and share a single aggregate revenue metric rather than channel-attributed returns, consider fitting jointly via `MultiChannelModel.fit_bayesian()` and calling `multi_model.get_allocator()`.
- [ ] **6. Interactive Stakeholder Simulation:** For live budget scenario workshops, launch the `dxpoint` interactive dashboard from the terminal:
  ```bash
  dxpoint dashboard
  ```